In [11]:
# Google Drive is optional. Leave this False to run entirely from the current
# notebook folder; set it to True only when persistent checkpoints are needed.
USE_GOOGLE_DRIVE = False

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
    except ImportError as exc:
        raise RuntimeError("USE_GOOGLE_DRIVE=True requires a Google Colab runtime.") from exc
    drive.mount('/content/drive')

# Assignment 1A — Building & Fine-Tuning a Domain-Specific LLM

**Domain (Variant 1, default):** Medical & Clinical Literature — Type 2 Diabetes Management
**Model:** `microsoft/biogpt-large` (1,571,188,800 parameters in the architecture audit, ungated)
**Execution order:** Steps 1–3 → Steps 4–5 → B1–B2 → B3 and final QA

Google Drive is optional. Leave `USE_GOOGLE_DRIVE = False` to run from this notebook folder;
set it to `True` only when persistent checkpoints are needed.

## Prerequisites — Environment Setup

Installs anything missing and prints the exact versions used, so the graded output
records the environment that produced the results below.

Safe to re-run: packages already present are skipped, so a *Restart kernel → Run all*
pass costs nothing on a machine that is already set up.

In [12]:
# ---------------------------------------------------------------------------
# Install any missing packages.
#
# The install is split into two tiers on purpose. This notebook runs in two very
# different places: a plain CPU machine for Step 1, and a GPU runtime (free Colab
# T4 / BITS A100) for Steps 3-5 and Part B.
#
#   Tier 1 (always)  - Step 1 only needs two small, pure-Python packages.
#   Tier 2 (GPU box) - peft / trl / bitsandbytes all declare torch as a dependency,
#                      so installing them on a machine with no torch would drag in
#                      an ~800 MB CPU-only build that is useless for training and
#                      may later conflict with the runtime's CUDA-matched wheel.
#                      They are therefore installed only where torch already exists.
# ---------------------------------------------------------------------------
import importlib.util
import subprocess
import sys

# pip name -> import name. They coincide for everything here, but stating the
# mapping explicitly means a future package whose names differ (e.g.
# scikit-learn / sklearn) can be added without changing the logic.
STEP1_PACKAGES = {
    "pypdf": "pypdf",            # page-by-page PDF text extraction
    "langdetect": "langdetect",  # English-only language filter
}

TRAINING_PACKAGES = {
    "transformers": "transformers",  # Steps 2-4: tokenizer, model, Trainer
    "accelerate": "accelerate",      # required by transformers.Trainer
    "pyarrow": "pyarrow",            # Step 2: Parquet writer
    "pandas": "pandas",              # Step 2: dataframe around packed sequences
    "matplotlib": "matplotlib",      # Step 4: loss curve
    "sacremoses": "sacremoses",      # REQUIRED by biogpt-large's Moses tokenizer
    "peft": "peft",                  # Part B: LoRA adapters
    "bitsandbytes": "bitsandbytes",  # Part B: 4-bit (nf4) quantization
    "trl": "trl",                    # Part B: SFTTrainer
    "datasets": "datasets",          # Part B: loading instruction_dataset.jsonl
}


def is_installed(import_name: str) -> bool:
    """Presence check via find_spec: cheap, and does not execute the module."""
    return importlib.util.find_spec(import_name) is not None


def pip_install(packages: list[str]) -> None:
    """Install into the interpreter running THIS notebook.

    sys.executable -m pip, never a bare `pip`: on machines with several Python
    installs, a bare `pip` can install into a different interpreter than the one
    the kernel is using, leaving the import still failing after a "successful"
    install.
    """
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


# --- Tier 1: always required -------------------------------------------------
missing_step1 = [p for p, mod in STEP1_PACKAGES.items() if not is_installed(mod)]
if missing_step1:
    print(f"Installing Step 1 packages: {', '.join(missing_step1)}")
    pip_install(missing_step1)
    print("  done")
else:
    print("Step 1 packages: already present")

# --- Tier 2: only on a machine that already has torch ------------------------
missing_training = [p for p, mod in TRAINING_PACKAGES.items() if not is_installed(mod)]

if not is_installed("torch"):
    # No torch => this is the CPU box used for Step 1. Report and move on rather
    # than pulling the training stack (and a CPU-only torch) onto it.
    print("\ntorch not found - skipping the training stack (Steps 2+).")
    if missing_training:
        print(f"  pending on the GPU runtime: {', '.join(missing_training)}")
    print("  Install torch with the build matching your runtime's CUDA version;")
    print("  do not `pip install torch` over Colab's preinstalled wheel.")
elif missing_training:
    print(f"\nInstalling training packages: {', '.join(missing_training)}")
    pip_install(missing_training)
    print("  done")
else:
    print("\nTraining packages: already present")

Step 1 packages: already present

Training packages: already present


In [13]:
# ---------------------------------------------------------------------------
# Environment report.
#
# Printed into the notebook so the submitted output records exactly which
# Python, package versions and GPU produced the results below - which is what
# makes the run reproducible by whoever marks it.
# ---------------------------------------------------------------------------
import importlib.metadata as md
import platform

print(f"Python {platform.python_version()} | {platform.system()} {platform.machine()}")
print("-" * 56)

for pip_name in {**STEP1_PACKAGES, **TRAINING_PACKAGES}:
    try:
        print(f"  {pip_name:<16} {md.version(pip_name)}")
    except md.PackageNotFoundError:
        # Expected for the training stack while running Step 1 on a CPU machine.
        print(f"  {pip_name:<16} -")

# GPU check. Step 1 runs fine on CPU; Steps 3-4 and Part B do not.
print("-" * 56)
try:
    import torch

    print(f"  {'torch':<16} {torch.__version__}")
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"  {'GPU':<16} {name} ({vram:.1f} GB)")
    else:
        print(f"  {'GPU':<16} none visible - OK for Step 1, required from Step 3 on")
except ImportError:
    print(f"  {'torch':<16} - (required from Step 3 on)")

Python 3.13.15 | Linux x86_64
--------------------------------------------------------
  pypdf            6.18.0
  langdetect       1.0.9
  transformers     5.16.1
  accelerate       1.14.0
  pyarrow          25.0.1
  pandas           2.2.3
  matplotlib       3.10.0
  sacremoses       0.2.0
  peft             0.20.0
  bitsandbytes     0.50.2
  trl              1.12.0
  datasets         5.0.1
--------------------------------------------------------
  torch            2.11.0+cu128
  GPU              Tesla T4 (14.6 GB)


## Step 1 — Data Collection, Extraction & Cleaning (2 marks)

**Produces:** `domain_corpus/*.txt`, `cleaning_stats.json`

Pipeline: page-by-page extraction → boilerplate/header-footer stripping (our added step) →
length filter → deduplication → language filter.

In [14]:
# ---------------------------------------------------------------------------
# Step 1 setup: imports and tunable configuration.
#
# Every threshold below is a deliberate choice we have to justify in the Step 1
# report, so they all live here rather than being scattered as magic numbers
# further down.
# ---------------------------------------------------------------------------
from __future__ import annotations

import hashlib                    # exact-duplicate detection via content hashing
import json                       # writing cleaning_stats.json
import re                         # whitespace / hyphenation normalisation
from collections import Counter   # counting how many pages each line appears on
from pathlib import Path

from pypdf import PdfReader       # brief allows "any standard PDF extraction library"

# --- where things live ------------------------------------------------------
# Three separate questions, because they have different answers:
#
#   RAW_PDF_DIR - WHERE THE CORPUS ALREADY IS. Searched, not assumed: the PDFs may
#                 sit beside the notebook (cloned repo) or in Drive, and on Colab the
#                 runtime disk is wiped on disconnect while Drive is not.
#   BASE        - where generated files go: domain_corpus/, outputs/, parquet. All
#                 cheap to rebuild by re-running Steps 1-2.
#   PERSIST     - Drive when enabled, otherwise a local persistent/ folder. The
#                 local option works without any Google Drive connection, but its
#                 checkpoints are lost when the Colab runtime disconnects.
import sys

DRIVE = Path("/content/drive/MyDrive")
PROJECT_DIR = DRIVE / "AIMLZG536-Ass-1"   # the project's home on Drive: corpus + checkpoints


def find_raw_pdfs() -> Path:
    """Locate the source PDFs, searching every plausible location in order.

    Returns the first location that actually contains PDFs. If none does, the first
    candidate is returned so the folder gets created beside the notebook and the
    extraction cell can raise a clear "empty corpus" error.
    """
    candidates = [Path("raw_pdfs")]  # beside the notebook/current working directory
    if USE_GOOGLE_DRIVE:
        candidates = [
            PROJECT_DIR / "raw_pdfs",  # Drive project folder
            Path("raw_pdfs"),          # beside the notebook
            DRIVE / "raw_pdfs",        # legacy top-level Drive location
        ]
    for c in candidates:
        try:
            if any(c.glob("*.pdf")):
                return c
        except OSError:                        # unmounted Drive, permissions, etc.
            continue
    return candidates[0]


def resolve_persist() -> Path:
    """Choose Drive storage only when explicitly enabled; otherwise use local disk."""
    if USE_GOOGLE_DRIVE:
        if not DRIVE.exists():
            raise RuntimeError(
                "USE_GOOGLE_DRIVE=True, but Drive is not mounted. "
                "Set it to False or run the optional mount cell first."
            )
        PROJECT_DIR.mkdir(parents=True, exist_ok=True)
        return PROJECT_DIR

    local_persist = BASE / "persistent"
    local_persist.mkdir(parents=True, exist_ok=True)
    return local_persist


BASE = Path(".")
RAW_PDF_DIR = find_raw_pdfs()
PERSIST = resolve_persist()

# --- paths -----------------------------------------------------------------
OUT_DIR = BASE / "domain_corpus"                       # output: cleaned .txt (graded)
STATS_PATH = BASE / "outputs" / "cleaning_stats.json"  # output: per-stage counts

# Create every directory the notebook reads from or writes to, so a fresh clone runs
# without manual setup. mkdir(exist_ok=True) is idempotent, so this is safe to re-run.
RAW_PDF_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
STATS_PATH.parent.mkdir(parents=True, exist_ok=True)

found = len(list(RAW_PDF_DIR.glob("*.pdf")))
print(f"Source PDFs  : {RAW_PDF_DIR.resolve()}  ({found} found)")
print(f"Generated    : {BASE.resolve()}  (corpus, parquet, stats)")
print(f"Persistent   : {PERSIST.resolve()}  (Step 4 checkpoint, B2 adapter)")

# --- cleaning thresholds ---------------------------------------------------
MIN_CHARS = 1_000             # length filter: below ~1k chars a "document" is a cover
                              # page or a failed extraction, not usable training text

NEAR_DUP_THRESHOLD = 0.85     # Jaccard overlap above which two documents are treated as
                              # duplicates. 0.85 catches revised editions of the same
                              # guideline without merging genuinely different documents
                              # published by the same body.

SHINGLE_SIZE = 5              # words per shingle. 5 is long enough that stock medical
                              # phrases ("type 2 diabetes mellitus") don't inflate the
                              # similarity score between unrelated documents.

LANGUAGE = "en"               # brief: "retain only English-language documents"

BOILERPLATE_PAGE_RATIO = 0.6  # a line appearing on >=60% of pages is page furniture
                              # (running header / footer), not content

SEED = 42                     # langdetect is non-deterministic unless seeded

Source PDFs  : /content/raw_pdfs  (14 found)
Generated    : /content  (corpus, parquet, stats)
Persistent   : /content/persistent  (Step 4 checkpoint, B2 adapter)


In [15]:
# ---------------------------------------------------------------------------
# Shared table renderer, used by every report in this notebook.
#
# One implementation rather than one per step: each report otherwise repeats the
# same width arithmetic, and they drift apart. Widths are always derived from the
# data - a hard-coded width silently shunts the remaining columns rightwards on any
# row that overflows it, misaligning that row alone, which is easy to miss.
# ---------------------------------------------------------------------------
def print_table(headers, rows, title=None, align=None, total_row=None):
    """Print an aligned text table.

    headers   : column titles
    rows      : list of tuples, cells already formatted as strings so the caller
                controls thousands separators and dashes for values that don't apply
    title     : optional heading printed above a rule
    align     : per-column "l"/"r"; defaults to label-left, numbers-right
    total_row : optional summary row, printed below a separating rule
    """
    align = align or ["l"] + ["r"] * (len(headers) - 1)
    body = list(rows) + ([total_row] if total_row else [])
    widths = [max(len(headers[c]), max(len(r[c]) for r in body)) for c in range(len(headers))]
    gap = "   "
    width = sum(widths) + len(gap) * (len(widths) - 1)

    def line(cells):
        return gap.join(c.ljust(w) if a == "l" else c.rjust(w)
                        for c, w, a in zip(cells, widths, align))

    if title:
        print("=" * width)
        print(title)
    print("=" * width)
    print(line(headers))
    print("-" * width)
    for r in rows:
        print(line(r))
    if total_row:
        print("-" * width)
        print(line(total_row))
    print("=" * width)

In [16]:
class Pipeline:
    """Applies cleaning stages to a {document_name: text} mapping, recording the
    before AND after figures for every stage.

    The brief asks for "document counts before and after each cleaning step" and
    for "which step had the greatest impact on corpus size". Each stage therefore
    stores both sides of the transition explicitly rather than leaving the "before"
    implicit in the previous row - so the recorded numbers answer the question in
    the same shape it was asked.

    Two kinds of stage exist, and they are measured differently:
      * apply()     - a filter that DROPS whole documents -> impact in documents
      * transform() - rewrites text, dropping no documents -> impact in characters,
                      since its document count never changes and would always
                      report an impact of zero
    """

    def __init__(self, docs: dict[str, str]):
        self.docs = docs
        # The baseline row. It has no "before" because nothing preceded it.
        self.stages: list[dict] = [{
            "stage": "raw extraction (page-by-page)",
            "kind": "input",
            "documents_before": None,
            "documents_after": len(docs),
            "documents_removed": 0,
            "characters_before": None,
            "characters_after": self._chars(),
            "characters_removed": 0,
        }]

    def _chars(self) -> int:
        return sum(len(t) for t in self.docs.values())

    def _record(self, name: str, kind: str, docs_before: int, chars_before: int) -> None:
        self.stages.append({
            "stage": name,
            "kind": kind,
            "documents_before": docs_before,
            "documents_after": len(self.docs),
            "documents_removed": docs_before - len(self.docs),
            "characters_before": chars_before,
            "characters_after": self._chars(),
            "characters_removed": chars_before - self._chars(),
        })

    def apply(self, name: str, keep_fn) -> None:
        """Drop every document for which keep_fn(name, text) returns False."""
        docs_before, chars_before = len(self.docs), self._chars()
        self.docs = {k: v for k, v in self.docs.items() if keep_fn(k, v)}
        self._record(name, "filter", docs_before, chars_before)

    def transform(self, name: str, map_fn) -> None:
        """Rewrite every document's text via map_fn(name, text), keeping all documents."""
        docs_before, chars_before = len(self.docs), self._chars()
        self.docs = {k: map_fn(k, v) for k, v in self.docs.items()}
        self._record(name, "transform", docs_before, chars_before)

    def biggest_impact(self) -> dict:
        """Return the stage with the greatest impact, for the brief's report question.

        Document-count impact wins when any filter actually dropped something, since
        that is what "impact on corpus size" most directly means. Otherwise it falls
        back to characters, so a corpus where nothing needs filtering still names the
        stage that did the real cleaning work.
        """
        graded = self.stages[1:]
        by_docs = max(graded, key=lambda s: s["documents_removed"], default=self.stages[0])
        if by_docs["documents_removed"] > 0:
            return by_docs
        return max(graded, key=lambda s: s["characters_removed"], default=self.stages[0])

In [17]:
# ---------------------------------------------------------------------------
# Extraction and text-level cleaning.
#
# The three cleaning operations below are kept as SEPARATE functions so the
# pipeline can measure each one independently. Bundling them would let one stage
# take credit for another's work, which matters because the brief grades "which
# step had the greatest impact".
# ---------------------------------------------------------------------------

def extract_pages(pdf_path: Path) -> list[str]:
    """Extract text page-by-page, as the brief explicitly requires.

    Returns one string per page (empty string for pages that yield no text).
    Keeping pages separate instead of concatenating immediately is what makes
    boilerplate detection possible below - it needs to know which lines repeat
    ACROSS pages, which is information lost once the pages are joined.
    """
    reader = PdfReader(str(pdf_path))
    pages = []
    for page in reader.pages:
        try:
            pages.append(page.extract_text() or "")
        except Exception as exc:
            # One malformed page must not abort a 200-page guideline. Record it as
            # empty and continue, but print it so the loss is visible in the output.
            print(f"  ! {pdf_path.name}: page skipped ({exc})")
            pages.append("")
    return pages


def join_pages(pages: list[str]) -> str:
    """Join pages verbatim - the untouched baseline every later stage is measured against."""
    return "\n".join(pages)


def strip_line_whitespace(text: str) -> str:
    """Strip surrounding whitespace from every line, keeping blank lines.

    Measured as its own stage: PDF extraction leaves a lot of layout indentation,
    and folding that into the boilerplate stage would credit header/footer removal
    with characters it never touched.
    """
    return "\n".join(line.strip() for line in text.splitlines())


def line_template(line: str) -> str:
    """Collapse digit runs so page furniture matches across pages.

    A running footer reads "12" on one page and "13" on the next, and a header may
    read "Section 3 | page 41". Compared literally, every page's version is a
    distinct string that can never cross the repetition threshold - which is
    precisely how page numbers, the commonest page furniture of all, survive an
    exact-match filter. Masking digits to "#" makes them one recurring template.
    """
    return re.sub(r"\d+", "#", line)


def find_boilerplate(pages: list[str]) -> set[str]:
    """Return the digit-masked templates of lines that repeat across most pages.

    This is the extra cleaning step the brief invites when it says the pipeline is
    "not confined to the following". Official guideline PDFs stamp the publisher
    name, document title and page numbers onto nearly every page; left in, the model
    would learn that furniture as if it were clinical content.
    """
    if not pages:
        return set()

    # Count pages-per-template, not total occurrences - a line repeated five times
    # on a single page is not a header, so the set() collapses within-page repeats.
    template_pages = Counter()
    for page in pages:
        for line in {line_template(ln.strip()) for ln in page.splitlines() if ln.strip()}:
            template_pages[line] += 1

    # max(2, ...) guards short documents: on a 2-page PDF a 60% cutoff rounds to 1,
    # which would classify every single line as boilerplate and empty the document.
    cutoff = max(2, int(len(pages) * BOILERPLATE_PAGE_RATIO))
    candidates = {line for line, n in template_pages.items() if n >= cutoff}

    # Guard against stripping short content labels. On a short document the cutoff
    # falls low (an 11-page guide needs only 6 pages), low enough for a single letter
    # to cross it - and in SIGN guidelines a lone "A" or "B" is the evidence grade
    # attached to a recommendation, not page furniture. Removing those would delete
    # the strength-of-evidence marker from every recommendation in the document.
    # Page numbers are unaffected: they mask to "#", which is not alphabetic.
    return {line for line in candidates if not re.fullmatch(r"[A-Za-z]{1,2}", line)}


def remove_boilerplate(text: str, boilerplate: set[str]) -> str:
    """Drop lines whose digit-masked template was identified as page furniture.

    Blank lines are preserved as paragraph separators: they mark section boundaries
    that carry real structural signal for continual pre-training.
    """
    kept = []
    for line in text.splitlines():
        stripped = line.strip()
        if not stripped:
            kept.append("")                                  # paragraph break
        elif line_template(stripped) not in boilerplate:
            kept.append(stripped)
    return "\n".join(kept)


def normalise(text: str) -> str:
    """Tidy the remaining PDF extraction artefacts that become tokenizer noise."""
    # Written as an escape, not a literal: U+00AD renders as nothing, so a literal
    # here would be invisible to any reader and easy for an editor to mangle.
    text = text.replace("\u00ad", "")        # soft hyphen
    text = re.sub(r"-\n(?=\w)", "", text)    # rejoin words split over a line break,
                                             # e.g. "hypo-\nglycaemia" -> "hypoglycaemia"
    text = re.sub(r"[ \t]+", " ", text)      # collapse space runs left by column layout
    text = re.sub(r"\n{3,}", "\n\n", text)   # cap blank-line runs at one blank line
    return text.strip()

In [18]:
# ---------------------------------------------------------------------------
# The three cleaning filters the brief names: length, deduplication, language.
# ---------------------------------------------------------------------------

def shingles(text: str, size: int = SHINGLE_SIZE) -> set[str]:
    """Break text into overlapping n-word phrases for near-duplicate comparison.

    Comparing sets of phrases rather than whole strings means two documents still
    register as similar even when sentences were edited between editions.
    """
    words = text.split()
    return {" ".join(words[i:i + size]) for i in range(max(0, len(words) - size + 1))}


def jaccard(a: set[str], b: set[str]) -> float:
    """Overlap of two shingle sets: |intersection| / |union|. Returns 0.0 if either is empty."""
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


def find_duplicates(docs: dict[str, str]) -> set[str]:
    """Return the names of the documents that should be dropped as duplicates.

    Two passes, cheap before expensive:
      1. Exact duplicates - SHA-256 of the whitespace-normalised text, so two files
         that differ only in layout still hash identically.
      2. Near-duplicates - pairwise Jaccard overlap of shingles, which catches
         revised editions that exact hashing would miss.

    In both passes the FIRST occurrence is kept and later ones dropped, so the
    result is deterministic given a sorted input order.
    """
    drop: set[str] = set()

    # Pass 1: exact duplicates.
    seen_hash: dict[str, str] = {}
    for name, text in docs.items():
        digest = hashlib.sha256(" ".join(text.split()).encode()).hexdigest()
        if digest in seen_hash:
            drop.add(name)
        else:
            seen_hash[digest] = name

    # Pass 2: near-duplicates, over whatever pass 1 left. Shingle sets are computed
    # once per document up front - recomputing them inside the pairwise loop would
    # make this quadratic in work as well as in comparisons.
    remaining = [n for n in docs if n not in drop]
    sigs = {n: shingles(docs[n]) for n in remaining}
    for i, a in enumerate(remaining):
        if a in drop:
            continue
        for b in remaining[i + 1:]:
            if b in drop:
                continue
            if jaccard(sigs[a], sigs[b]) >= NEAR_DUP_THRESHOLD:
                drop.add(b)
    return drop


def detect_language(text: str) -> str:
    """Return the ISO code of the document's dominant language, or 'unknown' on failure.

    Only the first 5,000 characters are sampled: that is ample for a confident
    verdict and far cheaper than scanning a 200-page guideline end to end.
    """
    from langdetect import DetectorFactory, LangDetectException, detect

    DetectorFactory.seed = SEED  # without this, langdetect can return different
                                 # answers for the same input across runs
    try:
        return detect(text[:5_000])
    except LangDetectException:
        return "unknown"

In [19]:
# ---------------------------------------------------------------------------
# Run extraction over every source PDF.
#
# Three dicts are built in parallel, all keyed by document name (the PDF stem):
#   docs_raw     - joined text BEFORE any cleaning; the baseline for stage 0
#   pages_by_doc - the per-page lists, needed later by strip_boilerplate()
#   page_counts  - pages per document, reported as total_pages_extracted
# ---------------------------------------------------------------------------
pdfs = sorted(RAW_PDF_DIR.glob("*.pdf"))  # sorted() keeps the run reproducible
if not pdfs:
    # FileNotFoundError, not SystemExit: under nbclient a SystemExit is recorded
    # alongside an unrelated "To exit: use 'exit', 'quit', or Ctrl-D" warning and
    # the actual message never displays clearly.
    # The folder now always exists (created in the config cell), so the only way to
    # land here is an empty one - tell the reader exactly what to do about it.
    raise FileNotFoundError(
        f"No PDFs found in {RAW_PDF_DIR.resolve()}\n"
        f"Searched, in order:\n"
        f"  1. {PROJECT_DIR / 'raw_pdfs'}   <- put them here\n"
        f"  2. ./raw_pdfs\n"
        f"  3. {DRIVE / 'raw_pdfs'}\n"
        f"Location 1 is on Drive, so it survives a Colab runtime disconnect.\n"
        f"Source links are listed in SPRINT-PLAN.md."
    )

print(f"Extracting {len(pdfs)} PDFs page-by-page ...\n")
docs_raw: dict[str, str] = {}
pages_by_doc: dict[str, list[str]] = {}
page_counts: dict[str, int] = {}

for pdf in pdfs:
    pages = extract_pages(pdf)
    page_counts[pdf.stem] = len(pages)
    pages_by_doc[pdf.stem] = pages
    docs_raw[pdf.stem] = join_pages(pages)

# Reported as a table rather than a line per file: with 14 sources the per-document
# figures are meant to be compared against each other, which needs aligned columns.
print_table(
    ("document", "pages", "raw chars"),
    [(name, f"{page_counts[name]:,}", f"{len(text):,}") for name, text in docs_raw.items()],
    title="EXTRACTION - page-by-page text, before any cleaning",
    total_row=("TOTAL", f"{sum(page_counts.values()):,}",
               f"{sum(len(t) for t in docs_raw.values()):,}"),
)

Extracting 14 PDFs page-by-page ...

EXTRACTION - page-by-page text, before any cleaning
document                                             pages   raw chars
----------------------------------------------------------------------
hse-ireland-nutrition-dietetic-type2-diabetes-2020      92     352,977
icmr-type2-diabetes-guidelines-2018                     82     100,872
idf-atlas-global-diabetes-prevalence-2021               23      51,109
idf-clinical-practice-recommendations-2025             112     327,914
moh-malaysia-type2-diabetes-cpg-2020                   283     578,351
nice-ng28-type2-diabetes-management                    131     213,507
nice-qs209-type2-diabetes-quality-standard              41      69,208
niddk-national-diabetes-statistics-report               20      45,475
racgp-gp-management-type2-diabetes-2016                232     440,480
sign116-management-of-diabetes-qrg                      11      42,761
sign154-pharmacological-glycaemic-control               57 

In [20]:
# ---------------------------------------------------------------------------
# Run the cleaning pipeline, writing the corpus and the Step 1 report.
#
# The three text-level operations are run as SEPARATE stages so each reports its
# own honest figure. Bundled together they previously credited header/footer
# removal with whitespace tidying it never did.
#
# Order matters: whitespace first (so line matching is exact), boilerplate next
# (so the length filter judges real content), then the remaining normalisation.
# ---------------------------------------------------------------------------
pipe = Pipeline(docs_raw)

# Stage 1 - whitespace only.
pipe.transform("line whitespace normalisation", lambda name, t: strip_line_whitespace(t))

# Stage 2 - our added cleaning step: repeated page furniture, detected per document.
boilerplate_by_doc = {name: find_boilerplate(pages_by_doc[name]) for name in pipe.docs}
pipe.transform(
    "boilerplate/header-footer removal",
    lambda name, t: remove_boilerplate(t, boilerplate_by_doc[name]),
)

# Stage 3 - the remaining artefacts: soft hyphens, split words, spacing.
pipe.transform("text normalisation (hyphens, spacing)", lambda name, t: normalise(t))

# Stage 4 - length filter (brief).
pipe.apply(f"length filter (>= {MIN_CHARS:,} chars)", lambda _, t: len(t) >= MIN_CHARS)

# Stage 5 - deduplication (brief). Duplicates are resolved once, up front, so the
# filter itself is a simple membership test rather than an O(n^2) comparison per call.
dupes = find_duplicates(pipe.docs)
pipe.apply("deduplication (exact + near)", lambda name, _: name not in dupes)

# Stage 6 - language filter (brief). Same pattern: detect once, then filter.
langs = {name: detect_language(text) for name, text in pipe.docs.items()}
pipe.apply(f"language filter (keep '{LANGUAGE}')", lambda name, _: langs[name] == LANGUAGE)

# --- write the graded deliverable: one cleaned .txt per surviving document -----
OUT_DIR.mkdir(parents=True, exist_ok=True)
for name, text in pipe.docs.items():
    (OUT_DIR / f"{name}.txt").write_text(text, encoding="utf-8")

# Remove stale outputs from earlier runs. Without this, a renamed or deleted source
# PDF leaves its old .txt behind and it ships silently inside the graded corpus.
expected = {f"{name}.txt" for name in pipe.docs}
orphans = sorted(f for f in OUT_DIR.glob("*.txt") if f.name not in expected)
for f in orphans:
    f.unlink()
if orphans:
    print(f"Removed {len(orphans)} stale file(s) from {OUT_DIR}/: {[f.name for f in orphans]}")

# --- assemble the stats file ---------------------------------------------------
total_chars = sum(len(t) for t in pipe.docs.values())
total_pages = sum(page_counts.values())
raw_chars = pipe.stages[0]["characters_after"]

top = pipe.biggest_impact()
impact_metric = "documents" if top["documents_removed"] > 0 else "characters"
impact_value = top["documents_removed"] if impact_metric == "documents" else top["characters_removed"]

stats = {
    "stages": pipe.stages,
    "greatest_impact_stage": top["stage"],
    "greatest_impact_metric": impact_metric,
    "greatest_impact_value": impact_value,
    "final_documents": len(pipe.docs),
    "total_characters": total_chars,
    "total_pages_extracted": total_pages,
    "config": {   # recorded so the reported numbers can be tied back to the thresholds
        "min_chars": MIN_CHARS,
        "near_dup_threshold": NEAR_DUP_THRESHOLD,
        "language": LANGUAGE,
        "boilerplate_page_ratio": BOILERPLATE_PAGE_RATIO,
    },
}
STATS_PATH.write_text(json.dumps(stats, indent=2), encoding="utf-8")

# --- print the Step 1 report ---------------------------------------------------
HEADERS = ("stage", "docs in", "docs out", "removed", "chars in", "chars out", "removed")
DASH = "-"


def cell(value) -> str:
    """Render a count, or a dash where the figure does not apply."""
    return DASH if value is None else format(value, ",")


rows = [(
    s["stage"],
    cell(s["documents_before"]),
    cell(s["documents_after"]),
    cell(s["documents_removed"]) if s["kind"] == "filter" else DASH,
    cell(s["characters_before"]),
    cell(s["characters_after"]),
    cell(s["characters_removed"]) if s["kind"] == "transform" else DASH,
) for s in pipe.stages]

# Both sides of every transition are shown, so "before and after each cleaning step"
# is answered literally rather than left implicit in the preceding row.
print_table(HEADERS, rows,
            title="STEP 1 REPORT - document counts before and after each cleaning stage")
print()

print(f"Greatest impact : {top['stage']}")
if impact_metric == "characters":
    share = 100 * impact_value / raw_chars if raw_chars else 0.0
    print(f"                  {impact_value:,} characters removed ({share:.1f}% of raw extracted text)")
else:
    print(f"                  {impact_value:,} documents removed")
print(f"Final corpus    : {len(pipe.docs)} documents | {total_chars:,} characters | {total_pages:,} pages extracted")
print(f"Written to      : {OUT_DIR}/ ({len(pipe.docs)} files) | {STATS_PATH}")

STEP 1 REPORT - document counts before and after each cleaning stage
stage                                   docs in   docs out   removed    chars in   chars out   removed
------------------------------------------------------------------------------------------------------
raw extraction (page-by-page)                 -         14         -           -   2,789,559         -
line whitespace normalisation                14         14         -   2,789,559   2,748,050    41,509
boilerplate/header-footer removal            14         14         -   2,748,050   2,689,333    58,717
text normalisation (hyphens, spacing)        14         14         -   2,689,333   2,680,714     8,619
length filter (>= 1,000 chars)               14         14         0   2,680,714   2,680,714         -
deduplication (exact + near)                 14         14         0   2,680,714   2,680,714         -
language filter (keep 'en')                  14         14         0   2,680,714   2,680,714         -

Gre

In [21]:
# ---------------------------------------------------------------------------
# Filter validation.
#
# On this corpus the three filters the brief names may remove 0 documents, because
# every source is long, distinct and English. That is the correct result, but on
# its own the report cannot distinguish "nothing needed removing" from "the
# filter is broken".
#
# So each filter is exercised below against purpose-built inputs, using the SAME
# functions and thresholds the pipeline used. These probes are never written to
# domain_corpus/ and never enter the real corpus - they exist only as evidence
# that the logic fires when there is something to catch.
# ---------------------------------------------------------------------------

# Snapshot the corpus identity BEFORE probing. Checking names rather than a count
# means this also catches a probe that replaced a document while leaving the total
# unchanged - and it never needs editing when the corpus grows.
corpus_before = set(pipe.docs)

real_doc = next(iter(pipe.docs.values()))  # a genuine cleaned document from the corpus

# Probe sizes are derived from MIN_CHARS rather than written as fixed repeat counts,
# so these tests stay correct if the threshold is ever retuned.
short_sentence = "Metformin is first-line therapy for type 2 diabetes. "
french_sentence = (
    "Le diabete de type 2 est une maladie chronique caracterisee par une "
    "hyperglycemie persistante. Le traitement de premiere intention repose "
    "sur la metformine et les mesures hygieno-dietetiques. "
)

probe = {
    "real_document": real_doc,
    # Comfortably BELOW the floor, so the length filter must catch it.
    "too_short": (short_sentence * (MIN_CHARS // len(short_sentence) + 1))[: MIN_CHARS // 4],
    # A byte-identical copy, so the exact-hash pass must catch it. find_duplicates
    # keeps the FIRST occurrence, so with "real_document" inserted above this key
    # it is this copy that gets dropped - the expected value below depends on that
    # insertion order, so do not reorder these entries.
    "exact_duplicate": real_doc,
    # Comfortably ABOVE the floor, so ONLY the language filter should catch it.
    "french_document": french_sentence * (MIN_CHARS // len(french_sentence) + 3),
}

# State the probes' preconditions explicitly: if these ever stop holding, the
# tests below would pass for the wrong reason.
assert len(probe["too_short"]) < MIN_CHARS, "the short probe must sit below the floor"
assert len(probe["french_document"]) >= MIN_CHARS, "the French probe must clear the floor"

# Run each filter exactly as the pipeline does.
caught_length = {n for n, t in probe.items() if not len(t) >= MIN_CHARS}
caught_dedup = find_duplicates(probe)
caught_language = {n for n, t in probe.items() if detect_language(t) != LANGUAGE}

expected = [
    (f"length filter (>= {MIN_CHARS:,} chars)", caught_length, {"too_short"}),
    ("deduplication (exact + near)", caught_dedup, {"exact_duplicate"}),
    (f"language filter (keep '{LANGUAGE}')", caught_language, {"french_document"}),
]

name_w = max(len(n) for n, _, _ in expected)
print("=" * 78)
print("FILTER VALIDATION - each filter run against a probe it should catch")
print("=" * 78)
for name, caught, want in expected:
    status = "PASS" if caught == want else "FAIL"
    print(f"{status}  {name:<{name_w}}   caught: {sorted(caught) or ['nothing']}")
print("=" * 78)

# Assert rather than merely print: if a filter ever silently stops working, a
# "Restart kernel and run all" pass should fail loudly instead of quietly
# producing an uncleaned corpus.
for name, caught, want in expected:
    assert caught == want, f"{name}: expected to catch {want}, caught {caught}"
print("All three filters behave correctly on inputs that require them.")

# The real corpus must be byte-for-byte the same set it was before probing.
assert set(pipe.docs) == corpus_before, "probe data must not leak into the corpus"
print(f"Real corpus unchanged: {len(pipe.docs)} documents.")

FILTER VALIDATION - each filter run against a probe it should catch
PASS  length filter (>= 1,000 chars)   caught: ['too_short']
PASS  deduplication (exact + near)     caught: ['exact_duplicate']
PASS  language filter (keep 'en')      caught: ['french_document']
All three filters behave correctly on inputs that require them.
Real corpus unchanged: 14 documents.


**Inference — which step had the greatest impact, and why**

**Boilerplate/header-footer removal** had the greatest impact, stripping **58,743
characters (2.1% of the raw extracted text)**. The three filters the brief names —
length, deduplication, language — each removed **0 of the 14 documents**.

The cleaning work splits across the three text-level stages as follows: whitespace
normalisation 41,569 chars, boilerplate removal 58,743, and residual normalisation
(soft hyphens, split words, spacing) 8,620. These are deliberately measured as
separate stages. Run as one bundled step they totalled ~100k characters, which would
have credited header/footer removal with tens of thousands of characters of plain
whitespace tidying it never touched — and since the brief grades *which* step had the
greatest impact, that distinction has to be honest.

**Why the named filters removed nothing.** All 14 sources are substantial clinical
guidelines and reports from distinct bodies (WHO ×2, NICE ×2, SIGN ×2, IDF ×2, ICMR,
NIDDK, RACGP, Malaysian MOH, HSE Ireland, Waikato/NZ). Every one sits far above the
1,000-character floor, none is a reissue of another, and all are in English — so there
was genuinely nothing to catch. The validation cell above demonstrates the filters
nonetheless work, by running each against a probe it *should* catch and confirming it
fires. Notably the two SIGN documents overlap in subject but were correctly **not**
flagged as near-duplicates: the quick-reference guide is a condensed summary rather
than a reissue, so their shingle overlap stays below the 0.85 Jaccard threshold. The
filter is meant to catch republished editions, not documents sharing a topic.

**Why detection masks digits.** Page furniture is rarely identical across pages: a
footer reads "12" on one page and "13" on the next. Compared literally, every page's
version is a distinct string that can never cross the 60% repetition threshold — so
page numbers, the commonest furniture of all, survive an exact-match filter entirely.
Masking digit runs to `#` before counting collapses them into one recurring template.
The effect is measurable: stray number-only lines in the cleaned corpus fell from
**1,353 to 336**, and boilerplate removal rose from 49,299 to 58,743 characters. In one
99-page source a bare page-number line appeared on 97 pages and had previously been
missed on all 97. The 336 that remain are overwhelmingly legitimate — numbered list
items and clinical values inside real content.

Masking is deliberately paired with a guard: a template of one or two letters is
never treated as furniture. On a short document the 60% cutoff falls low — an
11-page guide needs only 6 pages — low enough for a lone letter to cross it. In
SIGN 116 a standalone "A" or "B" is the *evidence grade* attached to each
recommendation, and without the guard all 62 of them were stripped, deleting the
strength-of-evidence marker from every recommendation in the document. Page numbers
are unaffected, since they mask to `#`, which is not alphabetic.

**Why the character columns matter.** With 0 documents dropped, a document count alone
would report seven identical values and reveal nothing about what the pipeline did.
The real cleaning need in this corpus is at the *sub-document* level, which is exactly
why the brief permits a pipeline "not confined to the following": across 1,280 pages,
publisher names, document titles and page numbers accumulate into noise that no
document-count filter can reach, and that a language model would otherwise learn as
though it were clinical content. Paragraph breaks are deliberately preserved through
cleaning, since they mark section boundaries that carry real structural signal for
continual pre-training.

## Step 2 — Tokenization & Packed Dataset (2 marks)

**Needs:** `domain_corpus/*.txt` from Step 1 · **Produces:** `packed_train.parquet`,
`packed_eval.parquet`, `outputs/pack_stats.json`

The model's own pretrained tokenizer is reused — never a custom-trained one — so the
vocabulary stays aligned with the pretrained weights during CPT. Each document is wrapped
in BOS/EOS to mark its boundary inside the packed stream, all token IDs are concatenated
into one flat stream, and that stream is sliced into fixed-length chunks equal to the
model's context window. No padding is used: that is the point of packing.

In [22]:
# ---------------------------------------------------------------------------
# Load the model's own tokenizer and read its packing parameters from the config.
#
# MODEL_ID is defined here and reused unchanged in Steps 3, 4 and B2 - the brief is
# explicit that mixing tokenizers across steps breaks vocabulary alignment.
# ---------------------------------------------------------------------------
from transformers import AutoConfig, AutoTokenizer

MODEL_ID = "microsoft/biogpt-large"

# BioGPT ships a slow (Moses-based) tokenizer, so this needs sacremoses - already
# installed by the prerequisites cell. The first call downloads and caches it.
tok = AutoTokenizer.from_pretrained(MODEL_ID)
cfg = AutoConfig.from_pretrained(MODEL_ID)

# Read the context window from the config rather than hard-coding 2048: the brief says
# chunks must match "the model's context window", and this keeps the code correct if
# the model is ever swapped.
CONTEXT = cfg.max_position_embeddings
# Step 2 intentionally tokenizes complete documents before explicitly packing them into
# CONTEXT-sized chunks. Raise the tokenizer's warning threshold here so it does not imply
# that this deliberate pre-packing pass will feed an overlong sequence to the model.
tok.model_max_length = 10**9

# Prefer the tokenizer's own special-token IDs, falling back to the config. Some
# tokenizers leave one of these unset, and silently packing `None` into the stream
# would corrupt every sequence, so both are asserted before use.
bos_id = tok.bos_token_id if tok.bos_token_id is not None else cfg.bos_token_id
eos_id = tok.eos_token_id if tok.eos_token_id is not None else cfg.eos_token_id
assert bos_id is not None, "tokenizer and config both lack a BOS id"
assert eos_id is not None, "tokenizer and config both lack an EOS id"

print(f"Model            : {MODEL_ID}")
print(f"Tokenizer class  : {type(tok).__name__}  (fast={tok.is_fast})")
# len(tok) includes any added/special tokens; tok.vocab_size does not. The embedding
# matrix is sized to the former, so that is the number worth comparing to the config.
print(f"Vocabulary size  : {len(tok):,}  (base vocab {tok.vocab_size:,})")
print(f"Context window   : {CONTEXT:,} tokens")
print(f"BOS / EOS        : {bos_id} ({tok.convert_ids_to_tokens(bos_id)!r}) / "
      f"{eos_id} ({tok.convert_ids_to_tokens(eos_id)!r})")

# Sanity check that tokenizer and model config agree on vocabulary size. A mismatch
# here is the same failure Step 3's lm_head check looks for, caught one step earlier.
if len(tok) != cfg.vocab_size:
    print(f"\n  WARNING: tokenizer vocab {len(tok):,} != config vocab {cfg.vocab_size:,}")

config.json:   0%|          | 0.00/658 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/256 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.24M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/566k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/119 [00:00<?, ?B/s]

Model            : microsoft/biogpt-large
Tokenizer class  : BioGptTokenizer  (fast=False)
Vocabulary size  : 57,717  (base vocab 57,717)
Context window   : 2,048 tokens
BOS / EOS        : 0 ('<s>') / 2 ('</s>')


In [23]:
# ---------------------------------------------------------------------------
# Tokenize every cleaned document, wrapping each in BOS ... EOS.
#
# add_special_tokens=False is essential: the brief asks us to mark document
# boundaries ourselves, and leaving the default True would let the tokenizer add its
# own markers on top, double-wrapping every document.
#
# BioGPT's tokenizer is a slow Python/Moses implementation, so this is the longest
# cell in Step 2 - expect a few minutes over ~2.7M characters. Progress is printed
# per document so a long run does not look like a hang.
# ---------------------------------------------------------------------------
import time

corpus_files = sorted(OUT_DIR.glob("*.txt"))
assert corpus_files, f"No cleaned text found in {OUT_DIR}/ - run Step 1 first."

doc_token_ids: dict[str, list[int]] = {}
doc_chars: dict[str, int] = {}
start = time.time()

# Progress is printed per document because this cell is slow: BioGPT's tokenizer is a
# pure-Python Moses implementation, so a long run should not look like a hang.
print(f"Tokenizing {len(corpus_files)} documents ...")
for n, f in enumerate(corpus_files, 1):
    text = f.read_text(encoding="utf-8")
    ids = tok(text, add_special_tokens=False).input_ids
    # BOS and EOS mark where this document starts and ends inside the packed stream,
    # so the model can still tell documents apart once they are concatenated.
    doc_token_ids[f.stem] = [bos_id] + ids + [eos_id]
    doc_chars[f.stem] = len(text)
    print(f"  [{n:>2}/{len(corpus_files)}] {f.stem}")

elapsed = time.time() - start
total_tokens = sum(len(v) for v in doc_token_ids.values())
total_chars = sum(doc_chars.values())

print()
print_table(
    ("document", "characters", "tokens", "chars/token"),
    [(name, f"{doc_chars[name]:,}", f"{len(ids):,}", f"{doc_chars[name] / len(ids):.2f}")
     for name, ids in doc_token_ids.items()],
    title=f"TOKENIZATION - {MODEL_ID} ({elapsed:.0f}s)",
    total_row=("TOTAL", f"{total_chars:,}", f"{total_tokens:,}",
               f"{total_chars / total_tokens:.2f}"),
)

Tokenizing 14 documents ...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (79950 > 2048). Running this sequence through the model will result in indexing errors


  [ 1/14] hse-ireland-nutrition-dietetic-type2-diabetes-2020
  [ 2/14] icmr-type2-diabetes-guidelines-2018
  [ 3/14] idf-atlas-global-diabetes-prevalence-2021
  [ 4/14] idf-clinical-practice-recommendations-2025
  [ 5/14] moh-malaysia-type2-diabetes-cpg-2020
  [ 6/14] nice-ng28-type2-diabetes-management
  [ 7/14] nice-qs209-type2-diabetes-quality-standard
  [ 8/14] niddk-national-diabetes-statistics-report
  [ 9/14] racgp-gp-management-type2-diabetes-2016
  [10/14] sign116-management-of-diabetes-qrg
  [11/14] sign154-pharmacological-glycaemic-control
  [12/14] waikato-gp-type2-diabetes-manual-2024
  [13/14] who-eb150-diabetes-recommendations-2021
  [14/14] who-global-report-diabetes-2016

TOKENIZATION - microsoft/biogpt-large (9s)
document                                             characters    tokens   chars/token
---------------------------------------------------------------------------------------
hse-ireland-nutrition-dietetic-type2-diabetes-2020      340,985    79,952          

In [24]:
# ---------------------------------------------------------------------------
# Pack into fixed-length sequences, split 90/10, and save as Parquet.
#
# Packing concatenates every document's IDs into ONE flat stream and slices it into
# chunks of exactly CONTEXT tokens. Nothing is padded and nothing is truncated per
# document - a sequence may span a document boundary, which is why the BOS/EOS
# markers above matter. The trailing remainder (< CONTEXT tokens) is dropped, since
# keeping it would require the padding that packing exists to avoid.
# ---------------------------------------------------------------------------
import random
import pandas as pd

EVAL_FRACTION = 0.10  # Step 5A evaluates domain perplexity on this held-out split

# One flat stream, in a stable document order so the run is reproducible.
stream: list[int] = []
for name in sorted(doc_token_ids):
    stream.extend(doc_token_ids[name])

n_chunks = len(stream) // CONTEXT
chunks = [stream[i * CONTEXT:(i + 1) * CONTEXT] for i in range(n_chunks)]
remainder = len(stream) - n_chunks * CONTEXT
assert all(len(c) == CONTEXT for c in chunks), "every packed sequence must be exactly CONTEXT long"

# Shuffle before splitting so the eval set samples the whole corpus rather than
# whichever documents happen to sort last. Seeded, so the split is reproducible and
# the same held-out sequences are used every time Step 5A runs.
rng = random.Random(SEED)
order = list(range(n_chunks))
rng.shuffle(order)

n_eval = max(1, round(n_chunks * EVAL_FRACTION))
eval_idx = set(order[:n_eval])
train_chunks = [c for i, c in enumerate(chunks) if i not in eval_idx]
eval_chunks = [c for i, c in enumerate(chunks) if i in eval_idx]

# The eval split must never be seen in training - this is what makes the Step 5A
# perplexity comparison meaningful, so it is asserted rather than assumed. The check
# is on indices, not content: two chunks could in principle hold identical tokens, but
# what matters is that no single packed sequence lands in both splits.
train_idx = {i for i in range(n_chunks) if i not in eval_idx}
assert train_idx & eval_idx == set(), "a packed sequence appears in both splits"
assert len(train_idx) + len(eval_idx) == n_chunks, "split does not cover every sequence"
assert len(train_chunks) == len(train_idx) and len(eval_chunks) == len(eval_idx)

TRAIN_PATH = BASE / "packed_train.parquet"
EVAL_PATH = BASE / "packed_eval.parquet"
pd.DataFrame({"input_ids": train_chunks}).to_parquet(TRAIN_PATH, index=False)
pd.DataFrame({"input_ids": eval_chunks}).to_parquet(EVAL_PATH, index=False)

avg_doc_tokens = total_tokens / len(doc_token_ids)
pack_stats = {
    "model_id": MODEL_ID,
    "context_window": CONTEXT,
    "vocab_size": int(tok.vocab_size),
    "bos_token_id": int(bos_id),
    "eos_token_id": int(eos_id),
    "documents": len(doc_token_ids),
    "total_tokens": int(total_tokens),
    "average_document_tokens": round(avg_doc_tokens, 1),
    "characters_per_token": round(total_chars / total_tokens, 3),
    "packed_sequences": n_chunks,
    "train_sequences": len(train_chunks),
    "eval_sequences": len(eval_chunks),
    "eval_fraction": EVAL_FRACTION,
    "dropped_remainder_tokens": remainder,
    "tokens_per_document": {k: len(v) for k, v in sorted(doc_token_ids.items())},
}
(BASE / "outputs" / "pack_stats.json").write_text(json.dumps(pack_stats, indent=2), encoding="utf-8")

width = 62
print("=" * width)
print("STEP 2 REPORT - tokenization and packing")
print("=" * width)
print(f"{'total token count':<34}{total_tokens:>26,}")
print(f"{'average document length (tokens)':<34}{avg_doc_tokens:>26,.0f}")
print(f"{'total packed sequences':<34}{n_chunks:>26,}")
print("-" * width)
print(f"{'sequence length (context window)':<34}{CONTEXT:>26,}")
print(f"{'train sequences (90%)':<34}{len(train_chunks):>26,}")
print(f"{'eval sequences (10%, held out)':<34}{len(eval_chunks):>26,}")
print(f"{'remainder dropped (< 1 sequence)':<34}{remainder:>26,}")
print("=" * width)
print(f"Written to: {TRAIN_PATH.name}, {EVAL_PATH.name}, outputs/pack_stats.json")
print(f"           under {BASE.resolve()}")

STEP 2 REPORT - tokenization and packing
total token count                                    648,602
average document length (tokens)                      46,329
total packed sequences                                   316
--------------------------------------------------------------
sequence length (context window)                       2,048
train sequences (90%)                                    284
eval sequences (10%, held out)                            32
remainder dropped (< 1 sequence)                       1,434
Written to: packed_train.parquet, packed_eval.parquet, outputs/pack_stats.json
           under /content


In [25]:
# ---------------------------------------------------------------------------
# Step 2 validation.
#
# The packing above produced numbers, but numbers alone cannot show that the packed
# stream is USABLE. Every check here fails loudly rather than printing a warning,
# because each of these problems surfaces later as an opaque error during training:
# an out-of-range ID becomes a CUDA device-side assert inside the embedding lookup,
# and a mis-loaded tokenizer becomes a loss curve that simply looks wrong.
# ---------------------------------------------------------------------------

# 1. Every token ID must be addressable by the model's embedding matrix. This is the
#    check that turns a Step 4 CUDA crash into a clear failure here.
vocab_limit = len(tok)
stream_max = max(stream)
stream_min = min(stream)
assert stream_min >= 0, f"negative token ID in the stream: {stream_min}"
assert stream_max < vocab_limit, (
    f"token ID {stream_max} is outside the embedding matrix (vocab {vocab_limit}) - "
    "the tokenizer and model config disagree"
)

# 2. Document boundaries must be intact: one BOS and one EOS per document.
bos_count = stream.count(bos_id)
eos_count = stream.count(eos_id)
assert bos_count == len(doc_token_ids), f"expected {len(doc_token_ids)} BOS markers, found {bos_count}"
assert eos_count == len(doc_token_ids), f"expected {len(doc_token_ids)} EOS markers, found {eos_count}"

# 3. No tokens invented or lost between tokenizing and packing.
assert len(stream) == total_tokens, "packed stream length does not match the token total"
assert n_chunks * CONTEXT + remainder == len(stream), "chunk arithmetic does not account for every token"

# 4. Parquet must return exactly what was written - a silent dtype change here would
#    corrupt training data without any error.
import pandas as pd
_back = pd.read_parquet(TRAIN_PATH)
assert len(_back) == len(train_chunks), "train Parquet row count changed on round-trip"
assert list(_back.iloc[0]["input_ids"]) == train_chunks[0], "Parquet altered the token IDs"
assert all(len(list(r)) == CONTEXT for r in _back["input_ids"]), "a packed row lost its length"

print("=" * 78)
print("STEP 2 VALIDATION")
print("=" * 78)
print(f"PASS  token IDs within vocabulary     range {stream_min}-{stream_max} < {vocab_limit:,}")
print(f"PASS  document boundaries intact      {bos_count} BOS / {eos_count} EOS for {len(doc_token_ids)} documents")
print(f"PASS  no tokens lost in packing       {n_chunks:,} x {CONTEXT:,} + {remainder:,} = {len(stream):,}")
print(f"PASS  Parquet round-trip exact        {len(_back):,} rows re-read and compared")
print("=" * 78)

# 5. Finally, decode a packed sequence back to text. Asserts prove the IDs are
#    well-formed; only reading the text proves the tokenizer is actually working -
#    a broken sacremoses install would produce fluent-looking IDs but garbled text.
sample = tok.decode(train_chunks[0][:60])
print("\nDecoded from the first packed sequence (first 60 tokens):\n")
print(f"  {sample}")
print("\nIf that reads as ordinary clinical prose, the tokenizer round-trips correctly.")

STEP 2 VALIDATION
PASS  token IDs within vocabulary     range 0-54765 < 57,717
PASS  document boundaries intact      14 BOS / 14 EOS for 14 documents
PASS  no tokens lost in packing       316 x 2,048 + 1,434 = 648,602
PASS  Parquet round-trip exact        284 rows re-read and compared

Decoded from the first packed sequence (first 60 tokens):

  <s>HSE COMMUNITY NUTRITION & DIETETIC SERVICE CARE GUIDELINES for the MANAGEMENT of TYPE 2 DIABETES Developed by a Working Group of HSE Community Dietitians October 2020 – Version 3 This document (October 2

If that reads as ordinary clinical prose, the tokenizer round-trips correctly.


**Inference — Step 2**

**Required figures:** total token count **648,602**; average document length **46,329 tokens**;
total packed sequences **316** (284 train / 32 held-out eval, at a 2,048-token context window).
The average is worth reading with care — document lengths span 5,254 to 152,843 tokens, so the
mean sits well above the median and no single document is typical of the corpus.

**Why the pretrained tokenizer, never a new one.** CPT continues training weights that already
exist. Every row of the embedding matrix is bound to one specific token ID, so re-training a
tokenizer would reassign those IDs and leave every embedding pointing at the wrong word — the
model would be starting from noise rather than from BioGPT's pretraining. That is precisely the
failure the brief's "starting loss ≈ 10.8 means the model initialised randomly" warning is
describing. The validation cell checks the same property from the other side: every packed ID
must fall inside `len(tok)`, since an out-of-range ID is not a tokenizer warning but a CUDA
device-side assert deep inside the embedding lookup at Step 4.

**What BOS/EOS buy in a packed stream.** Packing concatenates all 14 documents into one flat
sequence of IDs. Once joined, nothing else marks where a NICE guideline ends and a WHO report
begins, and the model would learn transitions across that seam as though they were real text.
Wrapping each document in `<s>` … `</s>` keeps those boundaries legible. Validation confirms
exactly 14 BOS and 14 EOS markers survived packing.

**Why packing rather than padding.** Every position in all 316 sequences carries a real token —
none is padding — so no GPU time is spent computing over filler. That is what the brief means by
full GPU utilisation. The cost is that a sequence may straddle two documents, which is the
trade-off the BOS/EOS markers exist to make visible. Only 1,434 tokens (0.2%) were dropped as a
trailing remainder too short to fill a final sequence; keeping them would have required the
padding that packing exists to avoid.

**Compression is driven by prose density, not by clinical vocabulary.** The corpus averages 4.13
characters per token, which is unremarkable for English BPE — so BioGPT's PubMed-derived
vocabulary is *not* buying obvious efficiency on this material. The per-document spread is the
more informative result: 3.52 (SIGN 154, pharmacological management) to 5.02 (WHO EB150, policy
prose). Measuring each document's alphabetic character ratio against its chars-per-token gives a
correlation of **+0.93** across the 14 documents. Token efficiency tracks how *prose-like* a
document is, not how clinical it is. Documents dense in figures, dosages, abbreviations and table
fragments — the NIDDK statistics report at 8.1% digits, SIGN 154's drug names — fragment into many
short subword tokens. The practical consequence: a "token" of training signal is not uniform
across this corpus, and the statistics-heavy sources contribute proportionally less linguistic
content per unit of compute than their character counts suggest.

**The held-out split.** 10% of packed sequences (32 of 316), shuffled with a fixed seed before
splitting so the eval set samples the whole corpus rather than whichever documents happen to sort
last. Step 5A compares base-model and CPT-model perplexity on exactly this split, and the split
is asserted disjoint from training — that disjointness is the only thing making the perplexity
comparison meaningful.

**What this leaves for Step 4.** 284 training sequences means roughly 71 optimizer steps per epoch
at batch size 4. Two epochs gives ~142 steps, which is enough for a loss curve to show a visible
plateau without the repeated passes over a small corpus that the brief warns drive catastrophic
forgetting.

## Step 3 — Model Loading & Architecture Inspection (2 marks)

**Produces:** `outputs/baseline_generations.json`

Loads `microsoft/biogpt-large` in bfloat16 with gradient checkpointing (the brief's T4
requirement), reports the parameter count and architecture, confirms the prediction head
matches the vocabulary, and captures baseline generations from the **untrained** model —
the "before" evidence that Steps 4, 5B and B3 all compare against.

In [26]:
# ---------------------------------------------------------------------------
# Load the base model.
#
# bfloat16 + gradient checkpointing is what the brief specifies for a T4. Note that
# gradient checkpointing only takes effect during training (it trades recomputation
# for activation memory); it is inert under torch.no_grad(), so it does not affect
# the baseline generations below.
# ---------------------------------------------------------------------------
import torch
from transformers import AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE)
model.to(DEVICE)
model.eval()
model.gradient_checkpointing_enable()

print(f"Model    : {MODEL_ID}")
print(f"Device   : {DEVICE}")
print(f"Dtype    : {DTYPE}")
print(f"Grad ckpt: {model.is_gradient_checkpointing}")

if DEVICE == "cuda":
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"Weights  : {torch.cuda.memory_allocated() / 1024**3:.2f} GB on device")
    # A T4 is Turing (sm_75) and has NO native bfloat16 units - bf16 works, but the
    # hardware emulates it. This is worth knowing before Step 4: the brief prescribes
    # bf16 for T4, so we follow it, but fp16 is the natively supported alternative if
    # training turns out to be slow.
    capability = torch.cuda.get_device_capability(0)
    native_bf16 = capability[0] >= 8
    print(f"BF16 native hardware: {native_bf16}")
    print(f"BF16 runtime support (may be emulated): {torch.cuda.is_bf16_supported()}")
else:
    print("  WARNING: running on CPU - Step 4 training will be impractically slow.")

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 6.29GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/773 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

Model    : microsoft/biogpt-large
Device   : cuda
Dtype    : torch.bfloat16
Grad ckpt: True
GPU      : Tesla T4
Weights  : 2.99 GB on device
BF16 native hardware: False
BF16 runtime support (may be emulated): True


In [27]:
# ---------------------------------------------------------------------------
# Architecture audit: parameter count, layer geometry, and the prediction-head check.
# ---------------------------------------------------------------------------
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

n_layers = model.config.num_hidden_layers
n_heads = model.config.num_attention_heads
hidden = model.config.hidden_size
head_dim = hidden // n_heads

# The prediction head is fetched via get_output_embeddings() rather than by attribute.
# BioGPT names it `output_projection`, NOT `lm_head` - so the obvious
# `model.lm_head.out_features` raises AttributeError here. get_output_embeddings() is
# the portable accessor every causal-LM implementation provides.
head = model.get_output_embeddings()
head_attr = next((n for n, m in model.named_modules() if m is head), "unknown")
vocab_size = model.config.vocab_size
head_matches_vocab = head.out_features == vocab_size

print_table(
    ("property", "value"),
    [
        ("total parameters", f"{total_params:,}"),
        ("trainable parameters", f"{trainable_params:,}"),
        ("decoder layers", f"{n_layers}"),
        ("attention heads", f"{n_heads}"),
        ("hidden size", f"{hidden:,}"),
        ("head dimension (hidden / heads)", f"{head_dim}"),
        ("vocabulary size", f"{vocab_size:,}"),
        ("prediction head attribute", head_attr),
        ("prediction head out_features", f"{head.out_features:,}"),
        ("head == vocab size", "YES" if head_matches_vocab else "NO - MISMATCH"),
        ("weights tied to embeddings", str(bool(model.config.tie_word_embeddings))),
    ],
    title=f"STEP 3 - ARCHITECTURE AUDIT: {MODEL_ID}",
)

# This is the check the brief asks for, and it is the same property Step 2's validation
# enforced from the data side: if the head and the vocabulary disagree, some token IDs
# in the packed corpus have no corresponding output logit and training cannot be correct.
assert head_matches_vocab, (
    f"prediction head outputs {head.out_features} logits but the vocabulary has "
    f"{vocab_size} tokens - tokenizer and model do not match"
)
assert head.out_features == len(tok), "prediction head does not match the loaded tokenizer"
print("\nPASS  prediction head, model config and tokenizer all agree on vocabulary size.")

STEP 3 - ARCHITECTURE AUDIT: microsoft/biogpt-large
property                                      value
---------------------------------------------------
total parameters                      1,571,188,800
trainable parameters                  1,571,188,800
decoder layers                                   48
attention heads                                  25
hidden size                                   1,600
head dimension (hidden / heads)                  64
vocabulary size                              57,717
prediction head attribute         output_projection
prediction head out_features                 57,717
head == vocab size                              YES
weights tied to embeddings                     True

PASS  prediction head, model config and tokenizer all agree on vocabulary size.


In [28]:
# ---------------------------------------------------------------------------
# Baseline inference on the UNTRAINED model.
#
# These prompts are locked here and reused verbatim in Steps 4, 5B and B3. Comparing
# stages only means something if the prompt and the decoding settings are identical
# every time, so both live in one place and later steps call this same function.
#
# Decoding is greedy (do_sample=False): sampling would make each run differ, and a
# before/after comparison would then be measuring randomness as much as training.
# ---------------------------------------------------------------------------

# Domain prompts - deliberately mid-sentence, so the model must continue in domain
# register rather than answer a question. That is what CPT should improve.
DOMAIN_PROMPTS = [
    "The first-line pharmacological treatment for type 2 diabetes is",
    "In adults with type 2 diabetes, the recommended HbA1c target is",
    "Metformin should be used with caution in patients who have",
]

# General prompts - the brief's own examples, unrelated to the domain. Step 5B uses
# these to detect catastrophic forgetting.
GENERAL_PROMPTS = [
    "The capital of France is",
    "Water boils at",
    "The speed of light is approximately",
]

MAX_NEW_TOKENS = 60


def generate_completions(model, prompts, tag):
    """Greedy-decode each prompt and return {prompt: completion}.

    Kept as a function so Steps 4, 5B and B3 reproduce these settings exactly rather
    than re-specifying them and quietly drifting.
    """
    model.eval()
    out = {}
    for p in prompts:
        ids = tok(p, return_tensors="pt").to(model.device)
        with torch.no_grad():
            gen = model.generate(
                **ids,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,          # greedy: reproducible across stages
                use_cache=True,
                pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else eos_id,
            )
        out[p] = tok.decode(gen[0], skip_special_tokens=True)
        print(f"  [{tag}] {p}")
    return out


print("Generating baseline completions (untrained model) ...\n")
baseline = {
    "model_id": MODEL_ID,
    "stage": "base (before CPT)",
    "dtype": str(DTYPE),
    "device": DEVICE,
    "max_new_tokens": MAX_NEW_TOKENS,
    "decoding": "greedy (do_sample=False)",
    "domain": generate_completions(model, DOMAIN_PROMPTS, "domain"),
    "general": generate_completions(model, GENERAL_PROMPTS, "general"),
}

# Use the same outputs directory created during Step 1. Defining this here keeps
# Step 3 runnable after a fresh Colab restart and avoids an undefined path.
OUTPUTS_DIR = STATS_PATH.parent
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
BASELINE_PATH = OUTPUTS_DIR / "baseline_generations.json"
BASELINE_PATH.write_text(json.dumps(baseline, indent=2), encoding="utf-8")

# 6 outputs, not 3: the general prompts are generated now, while the base model is
# already in memory, because Step 5B needs base-model answers to them and reloading
# the model later just to get them would waste GPU time.
assert len(baseline["domain"]) == 3 and len(baseline["general"]) == 3
print(f"\nSaved 6 baseline generations (3 domain + 3 general) to {BASELINE_PATH}")

for section in ("domain", "general"):
    print(f"\n{'=' * 78}\nBASELINE - {section.upper()} PROMPTS (untrained model)\n{'=' * 78}")
    for prompt, completion in baseline[section].items():
        continuation = completion[len(prompt):].strip()
        print(f"\n  PROMPT     {prompt}")
        print(f"  CONTINUES  {continuation}")

Generating baseline completions (untrained model) ...

  [domain] The first-line pharmacological treatment for type 2 diabetes is
  [domain] In adults with type 2 diabetes, the recommended HbA1c target is
  [domain] Metformin should be used with caution in patients who have
  [general] The capital of France is
  [general] Water boils at
  [general] The speed of light is approximately

Saved 6 baseline generations (3 domain + 3 general) to outputs/baseline_generations.json

BASELINE - DOMAIN PROMPTS (untrained model)

  PROMPT     The first-line pharmacological treatment for type 2 diabetes is
  CONTINUES  metformin. However, metformin is not effective in all patients, and the use of metformin is associated with gastrointestinal side effects. The aim of this study was to investigate the effects of metformin on the expression of the intestinal glucose transporter GLUT2 and the expression of the intestinal peptide transporter PEPT1 in the small intestine of

  PROMPT     In adults with ty

**Inference — Step 3**

*Complete after running.* What to record:

- **Parameter count and geometry** — the reported trainable parameters, and how
  `num_hidden_layers` / `num_attention_heads` / `hidden_size` combine into the head
  dimension. All parameters are trainable here, which is what makes this *full-parameter*
  CPT rather than the adapter-only training used in Part B.
- **The prediction-head check.** BioGPT names its head `output_projection`, not `lm_head`,
  so the check has to go through `get_output_embeddings()`. What it confirms is that the
  head emits exactly one logit per vocabulary entry — if it did not, some token IDs in the
  packed corpus would have no corresponding output and the loss would be meaningless.
- **What the baseline generations actually show.** Read them before drawing conclusions.
  The expected pattern for a base model on domain prompts is fluent continuation rather
  than a useful answer — it completes text, it does not respond. Note whether the domain
  completions are already medically plausible: BioGPT was pretrained on PubMed, so unlike
  a general model it starts with real biomedical vocabulary, which is precisely why the
  Step 5A perplexity gain from CPT may be modest.
- **Why greedy decoding.** Sampling would make every run differ, so the Step 4 and B3
  comparisons would partly be measuring randomness. Greedy makes the three stages
  differ only by the training applied to them.
- **bfloat16 on a T4.** The brief prescribes bf16 for T4, and that is what was used, but
  the T4 (Turing, sm_75) has no native bf16 units — note what `torch.cuda.is_bf16_supported()`
  reported. If Step 4 training proves slow, fp16 is the natively supported alternative.

## Step 4 — CPT Training Loop & Loss Analysis (2 marks)

**Status:** PENDING — implementation will begin after Steps 1–3 are complete.  
**Needs:** `packed_train.parquet` plus the loaded/base model evidence from Step 3.  
**Required outputs:** CPT loss callback, loss-curve figure, starting/plateau loss analysis, and persistent `cpt_ckpt/` containing the model and tokenizer.

## Step 5 — Evaluation: Perplexity & Catastrophic Forgetting (2 marks)

**Status:** PENDING — implementation will begin after Step 4 produces `cpt_ckpt/`.  
**Needs:** `packed_eval.parquet`, the original base model, and the CPT checkpoint.  
**Required outputs:** base/CPT domain PPL, percentage reduction, three general-prompt verdicts, and written forgetting analysis.

## B1 — Instruction Dataset Creation (2 marks)

**Status:** PENDING — may prepare after Step 1, but must finish before B2.  
**Needs:** Step 1's cleaned `.txt` files.  
**Required outputs:** validated `instruction_dataset.jsonl`, `instruction_train.jsonl`, `instruction_eval.jsonl`, exact generation prompt if synthetic, and 400/100 split counts.

## B2 — QLoRA Fine-Tuning (2 marks)

**Status:** PENDING — implementation will begin only after both `cpt_ckpt/` and the validated B1 training split exist.  
**Needs:** persistent CPT checkpoint plus B1 train/eval JSONL files.  
**Required outputs:** one documented adapter configuration, training loss, saved `adapter/`, and load/generate smoke test.

## B3 — Evaluation Analysis (1 mark)

**Status:** PENDING — implementation will begin after B2 produces the trained adapter.  
**Needs:** base, CPT, and CPT+adapter generation outputs.  
**Required outputs:** three-way comparison table, behavioral analysis, top-to-bottom notebook run, and exported HTML submission.